In [1]:
import random
import numpy as np
import pandas as pd

from neo4j import GraphDatabase
from getpass import getpass
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

In [2]:
# 1. Koneksi ke Neo4j

URI = "bolt://127.0.0.1:7687"
USERNAME = "neo4j"
PASSWORD = getpass("Masukkan password Neo4j: ")

In [4]:
# Jika menjalankan file dari folder utama proyek, ini akan menunjuk
# ke folder data/processed di dalam proyek tersebut.
PROJECT_DIR = Path(
    r"C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"

P1_FILE = DATA_DIR / "events_P1_training.csv"
P2_FILE = DATA_DIR / "events_P2_testing.csv"

print("Folder data:", DATA_DIR)
print("File P1 tersedia:", P1_FILE.exists())
print("File P2 tersedia:", P2_FILE.exists())

Folder data: C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed
File P1 tersedia: True
File P2 tersedia: True


In [5]:
SEED = 42
K = 5
JUMLAH_REKOMENDASI = 3
MIN_ITEM_TARGET = 2

In [ ]:
if not P1_FILE.exists():
    raise FileNotFoundError(
        f"File P1 tidak ditemukan: {P1_FILE}\n"
        "Pastikan file events_P1_training.csv berada di data/processed."
    )

if not P2_FILE.exists():
    raise FileNotFoundError(
        f"File P2 tidak ditemukan: {P2_FILE}\n"
        "Pastikan file events_P2_testing.csv berada di data/processed."
    )

p1 = pd.read_csv(P1_FILE)
p2 = pd.read_csv(P2_FILE)

In [7]:
# Dataset RetailRocket biasanya memakai visitorid dan itemid.
# Bagian ini juga menerima user_id dan item_id jika file sudah diubah namanya.
def standarkan_kolom(df, nama_file):
    df = df.copy()

    rename_map = {}
    if "visitorid" in df.columns:
        rename_map["visitorid"] = "user_id"
    if "itemid" in df.columns:
        rename_map["itemid"] = "item_id"

    df = df.rename(columns=rename_map)

    kolom_wajib = {"user_id", "item_id"}
    if not kolom_wajib.issubset(df.columns):
        raise ValueError(
            f"{nama_file} harus memiliki kolom visitorid/itemid "
            "atau user_id/item_id. "
            f"Kolom yang ditemukan: {list(df.columns)}"
        )

    df = df.dropna(subset=["user_id", "item_id"]).copy()
    return df

p1 = standarkan_kolom(p1, "P1")
p2 = standarkan_kolom(p2, "P2")

In [8]:
# Pastikan tipe ID konsisten antara P1 dan P2.
# RetailRocket umumnya menggunakan ID numerik.
for df in (p1, p2):
    df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce")
    df["item_id"] = pd.to_numeric(df["item_id"], errors="coerce")
    df.dropna(subset=["user_id", "item_id"], inplace=True)
    df["user_id"] = df["user_id"].astype("int64")
    df["item_id"] = df["item_id"].astype("int64")

# Gunakan pasangan user-item unik untuk matriks biner.
# Jika user berinteraksi berkali-kali dengan item yang sama,
# pasangan itu tetap dihitung sebagai satu interaksi.
p1_pairs = p1[["user_id", "item_id"]].drop_duplicates()

print("===== RINGKASAN DATA =====")
print(f"Baris P1 asli                 : {len(p1):,}")
print(f"Pasangan user-item unik P1    : {len(p1_pairs):,}")
print(f"Baris P2 asli                 : {len(p2):,}")
print(f"User unik P1                  : {p1_pairs['user_id'].nunique():,}")
print(f"Item unik P1                  : {p1_pairs['item_id'].nunique():,}")
print()

===== RINGKASAN DATA =====
Baris P1 asli                 : 333,778
Pasangan user-item unik P1    : 266,342
Baris P2 asli                 : 303,710
User unik P1                  : 185,205
Item unik P1                  : 87,308



In [9]:
user_ids = np.array(sorted(p1_pairs["user_id"].unique()), dtype=np.int64)
item_ids = np.array(sorted(p1_pairs["item_id"].unique()), dtype=np.int64)

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

rows = p1_pairs["user_id"].map(user_to_idx).to_numpy()
cols = p1_pairs["item_id"].map(item_to_idx).to_numpy()
values = np.ones(len(p1_pairs), dtype=np.float32)

X = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32
)

print("===== MATRIKS VSM USER-ITEM =====")
print(f"Shape matriks : {X.shape}")
print(f"Nonzero       : {X.nnz:,}")
print()

# Riwayat item tiap user, berdasarkan P1 saja.
riwayat_user = (
    p1_pairs.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

===== MATRIKS VSM USER-ITEM =====
Shape matriks : (185205, 87308)
Nonzero       : 266,342



In [10]:
# Untuk dapat dievaluasi pada 2.2.2, target harus muncul di P2.
user_p2 = set(p2["user_id"].unique())

# Syarat awal:
# (1) punya minimal 2 item unik di P1,
# (2) muncul pada P2,
# (3) ada pada matriks P1.
kandidat_user = [
    int(user_id)
    for user_id in user_ids
    if len(riwayat_user.get(int(user_id), set())) >= MIN_ITEM_TARGET
    and int(user_id) in user_p2
]

if not kandidat_user:
    raise RuntimeError(
        "Tidak ada user yang memenuhi syarat: minimal 2 item unik di P1 "
        "dan memiliki interaksi di P2."
    )

rng = np.random.default_rng(SEED)
urutan_acak = rng.permutation(len(kandidat_user))

print("===== PEMILIHAN USER =====")
print(f"Seed                           : {SEED}")
print(f"Kandidat user yang memenuhi syarat: {len(kandidat_user):,}")
print("Kriteria: minimal 2 item unik pada P1 dan muncul pada P2.")
print()

===== PEMILIHAN USER =====
Seed                           : 42
Kandidat user yang memenuhi syarat: 986
Kriteria: minimal 2 item unik pada P1 dan muncul pada P2.



In [11]:
def rekomendasikan_untuk_user(target_user):
    """
    Mengembalikan tetangga terdekat dan kandidat rekomendasi.
    Rekomendasi hanya memakai interaksi P1.
    """
    target_idx = user_to_idx[target_user]
    target_vector = X[target_idx]

    # Cosine similarity target terhadap seluruh user pada P1.
    similarities = cosine_similarity(target_vector, X).ravel()

    # Keluarkan target dari daftar tetangga.
    similarities[target_idx] = -1.0

    # Tetangga hanya yang punya similarity positif.
    indeks_valid = np.flatnonzero(similarities > 0)

    if len(indeks_valid) == 0:
        return [], {}

    # Urutan deterministik: similarity terbesar dahulu,
    # lalu indeks user untuk memecahkan tie.
    urutan = sorted(
        indeks_valid,
        key=lambda idx: (-float(similarities[idx]), int(idx))
    )
    neighbor_indices = urutan[:K]

    neighbor_info = [
        (int(user_ids[idx]), float(similarities[idx]))
        for idx in neighbor_indices
    ]

    # Skor kandidat = jumlah cosine similarity tetangga yang berinteraksi
    # dengan item tersebut. Item yang sudah ada di riwayat target dikecualikan.
    item_target = riwayat_user[target_user]
    skor_item = {}

    for neighbor_user, similarity_score in neighbor_info:
        for item_id in riwayat_user.get(neighbor_user, set()):
            if item_id in item_target:
                continue
            skor_item[item_id] = (
                skor_item.get(item_id, 0.0) + similarity_score
            )

    return neighbor_info, skor_item


In [12]:
target_user = None
neighbor_info = None
skor_item = None

for posisi in urutan_acak:
    calon_target = kandidat_user[int(posisi)]
    calon_neighbors, calon_skor_item = rekomendasikan_untuk_user(calon_target)

    # Pastikan ada setidaknya 3 kandidat item baru.
    if len(calon_skor_item) >= JUMLAH_REKOMENDASI:
        target_user = calon_target
        neighbor_info = calon_neighbors
        skor_item = calon_skor_item
        break

if target_user is None:
    print("BELUM DITEMUKAN TARGET DENGAN 3 REKOMENDASI BARU.")
    print(
        "Artinya, pada kandidat yang memenuhi syarat, lima tetangga "
        "terdekat belum menyediakan cukup item baru. Jangan membuat "
        "hasil rekomendasi palsu; periksa distribusi interaksi atau "
        "pertimbangkan aturan tetangga yang berbeda dan jelaskan di laporan."
    )
    raise SystemExit(0)

# Ambil 3 item teratas: skor tertinggi, lalu item_id sebagai tie-break.
rekomendasi_teratas = sorted(
    skor_item.items(),
    key=lambda pasangan: (-pasangan[1], pasangan[0])
)[:JUMLAH_REKOMENDASI]

hasil_rekomendasi = pd.DataFrame(
    [
        {
            "peringkat": peringkat,
            "item_rekomendasi": int(item_id),
            "skor_knn": float(skor)
        }
        for peringkat, (item_id, skor)
        in enumerate(rekomendasi_teratas, start=1)
    ]
)

In [13]:
print("===== HASIL KNN RECOMMENDATION =====")
print(f"Seed                    : {SEED}")
print(f"User target terpilih    : {target_user}")
print(f"Jumlah item riwayat P1  : {len(riwayat_user[target_user])}")
print(f"Item pada riwayat target: {sorted(riwayat_user[target_user])}")
print()

print(f"Lima tetangga terdekat (k={K}):")
df_neighbors = pd.DataFrame(
    [
        {
            "neighbor_user": neighbor_user,
            "cosine_similarity": similarity_score
        }
        for neighbor_user, similarity_score in neighbor_info
    ]
)
print(df_neighbors.to_string(index=False))
print()

print(f"Tiga rekomendasi untuk user {target_user}:")
print(hasil_rekomendasi.to_string(index=False))
print()

===== HASIL KNN RECOMMENDATION =====
Seed                    : 42
User target terpilih    : 1271169
Jumlah item riwayat P1  : 2
Item pada riwayat target: [226111, 443507]

Lima tetangga terdekat (k=5):
 neighbor_user  cosine_similarity
        609849           0.707107
       1071056           0.707107
        191496           0.500000
       1256728           0.408248
        274208           0.353553

Tiga rekomendasi untuk user 1271169:
 peringkat  item_rekomendasi  skor_knn
         1            241284  0.500000
         2             81949  0.408248
         3            344054  0.408248



In [14]:
# Simpan hasil di folder data/processed.
output_rekomendasi = DATA_DIR / "rekomendasi_knn_k5.csv"
output_neighbors = DATA_DIR / "tetangga_knn_k5.csv"

hasil_rekomendasi.to_csv(output_rekomendasi, index=False)
df_neighbors.to_csv(output_neighbors, index=False)

# Simpan metadata target untuk dipakai saat evaluasi soal 2.2.2.
metadata = pd.DataFrame(
    [{
        "seed": SEED,
        "target_user": target_user,
        "k": K,
        "jumlah_item_riwayat_p1": len(riwayat_user[target_user]),
        "jumlah_kandidat_user": len(kandidat_user),
    }]
)
metadata.to_csv(DATA_DIR / "metadata_knn_k5.csv", index=False)

print("File hasil disimpan:")
print(f"- {output_rekomendasi}")
print(f"- {output_neighbors}")
print(f"- {DATA_DIR / 'metadata_knn_k5.csv'}")
print()
print("Catatan metodologi:")
print("- Matriks dan kemiripan dibangun dari interaksi P1 saja.")
print("- P2 hanya digunakan untuk memastikan target dapat dievaluasi.")
print("- Skor rekomendasi adalah jumlah cosine similarity dari tetangga")
print("  yang berinteraksi dengan item tersebut.")
print("- Seed dan aturan pemilihan target dicatat agar proses reproducible.")


File hasil disimpan:
- C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed\rekomendasi_knn_k5.csv
- C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed\tetangga_knn_k5.csv
- C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed\metadata_knn_k5.csv

Catatan metodologi:
- Matriks dan kemiripan dibangun dari interaksi P1 saja.
- P2 hanya digunakan untuk memastikan target dapat dievaluasi.
- Skor rekomendasi adalah jumlah cosine similarity dari tetangga
  yang berinteraksi dengan item tersebut.
- Seed dan aturan pemilihan target dicatat agar proses reproducible.


# **SKOR ITEM**

In [15]:
top5 = sorted(
    skor_item.items(),
    key=lambda x: (-x[1], x[0])
)[:5]

df_top5 = pd.DataFrame(
    [
        {
            "peringkat": rank,
            "item_id": int(item),
            "skor_knn": float(score)
        }
        for rank, (item, score) in enumerate(top5, start=1)
    ]
)

print("TOP-5 REKOMENDASI")
print(df_top5.to_string(index=False))

TOP-5 REKOMENDASI
 peringkat  item_id  skor_knn
         1   241284  0.500000
         2    81949  0.408248
         3   344054  0.408248
         4    21967  0.353553
         5    47940  0.353553


In [16]:
# Item yang diinteraksikan target pada periode P2
item_p2 = set(
    p2.loc[
        p2["user_id"] == target_user,
        "item_id"
    ].dropna().astype(int)
)

# Keluarkan item yang sudah ada di riwayat P1.
# Evaluasi berfokus pada item baru yang muncul pada P2.
item_p1 = set(riwayat_user[target_user])
item_relevan = item_p2 - item_p1

print("\nUSER TARGET:", target_user)
print("Jumlah item relevan di P2:", len(item_relevan))
print("Item relevan di P2:", sorted(item_relevan))


USER TARGET: 1271169
Jumlah item relevan di P2: 1
Item relevan di P2: [171924]


In [17]:
# Hit@5: apakah ada item relevan di antara 5 rekomendasi teratas?
daftar_top5 = [
    int(item) for item, score in top5
]

jumlah_hit = len(set(daftar_top5) & item_relevan)

hit_at_5 = 1 if jumlah_hit > 0 else 0

print("\nJUMLAH ITEM RELEVAN DI TOP-5:", jumlah_hit)
print("Hit@5 =", hit_at_5)


JUMLAH ITEM RELEVAN DI TOP-5: 0
Hit@5 = 0


In [18]:
# Relevansi biner: 1 jika item ditemukan pada P2,
# 0 jika tidak ditemukan.
relevansi = [
    1 if item in item_relevan else 0
    for item in daftar_top5
]

# DCG@5
dcg_at_5 = sum(
    rel / np.log2(rank + 1)
    for rank, rel in enumerate(relevansi, start=1)
)

# IDCG@5: urutan ideal dengan item relevan
# ditempatkan di peringkat tertinggi.
jumlah_relevan_top5 = min(len(item_relevan), 5)

idcg_at_5 = sum(
    1 / np.log2(rank + 1)
    for rank in range(1, jumlah_relevan_top5 + 1)
)

ndcg_at_5 = (
    dcg_at_5 / idcg_at_5
    if idcg_at_5 > 0
    else 0.0
)

print("\nRELEVANSI TOP-5:", relevansi)
print(f"DCG@5  = {dcg_at_5:.6f}")
print(f"IDCG@5 = {idcg_at_5:.6f}")


RELEVANSI TOP-5: [0, 0, 0, 0, 0]
DCG@5  = 0.000000
IDCG@5 = 1.000000


In [19]:
print("User target:", target_user)
print("Item relevan di P2:", sorted(item_relevan))
print("Jumlah item relevan di P2:", len(item_relevan))

print("\nTop-5 rekomendasi:", daftar_top5)
print("Relevansi top-5:", relevansi)
print("Jumlah hit:", jumlah_hit)

print(f"\nHit@5  = {hit_at_5}")
print(f"DCG@5  = {dcg_at_5:.6f}")
print(f"IDCG@5 = {idcg_at_5:.6f}")
print(f"NDCG@5 = {ndcg_at_5:.6f}")

User target: 1271169
Item relevan di P2: [171924]
Jumlah item relevan di P2: 1

Top-5 rekomendasi: [241284, 81949, 344054, 21967, 47940]
Relevansi top-5: [0, 0, 0, 0, 0]
Jumlah hit: 0

Hit@5  = 0
DCG@5  = 0.000000
IDCG@5 = 1.000000
NDCG@5 = 0.000000
